# PPR10K Setup (for ECCV rebuttal)

**核心思路**: 不用下载 91GB! PPR10K 全部托管在公开的 [Google Drive 文件夹](https://drive.google.com/drive/folders/1dKO1mKXCBbuE6KsZWPdMrjEjWJnJOY1k),**直接加 shortcut 到你的 Drive**,Colab 立刻能访问。零下载、零空间占用。

## 一次性手动步骤 (5 秒)

1. 浏览器打开 https://drive.google.com/drive/folders/1dKO1mKXCBbuE6KsZWPdMrjEjWJnJOY1k
2. **右键** `train_val_images_tif_360p` 文件夹 (91GB,360p 配对数据)
3. 点 **"Organize" → "Add shortcut"** (老 UI 是 "Add shortcut to Drive")
4. 选 **"My Drive" → "datasets"** 作为目标 → Add
5. (可选) 同样操作给 `masks_360p` 加 shortcut (只 56MB,如果 rebuttal 用 mask-aware loss 才需要)

做完后你的 `MyDrive/datasets/` 里会出现一个带 shortcut 标记的 `train_val_images_tif_360p`。
Drive 不会真复制 91GB,只是个引用。Colab 看不出区别。

---

## PPR10K 数据结构 (官方说明)

- 共 **11,161** 张人像
- Train/Val 切分: **前 8,875 张训练,后 2,286 张验证** (按 filename 排序)
- 每张图 360p 版本下有: `source` + `5 个 augmented sources` + `target_a` + `target_b` + `target_c`
- 我们做基本 paired retouching benchmark,**只用 `source` (无增强) + `target_<expert>`**

In [ ]:
# === Cell 1: mount Drive,确认 shortcut 已加 ===
from google.colab import drive
drive.mount('/content/drive')

import os, glob

PPR_ROOT = '/content/drive/MyDrive/datasets/train_val_images_tif_360p'
assert os.path.exists(PPR_ROOT), f"❌ {PPR_ROOT} 不存在 — 请先去 Drive 网页加 shortcut (见上面 markdown)"

print(f"✅ {PPR_ROOT} 存在")
print("\n=== 内部结构 ===")
for d in sorted(os.listdir(PPR_ROOT)):
    full = f'{PPR_ROOT}/{d}'
    if os.path.isdir(full):
        try:
            n = len(os.listdir(full))
            print(f"  📂 {d}/ ({n} files)")
        except Exception as e:
            print(f"  📂 {d}/ (无法列举: {e})")
    else:
        print(f"  📄 {d}")

In [ ]:
# === Cell 2: 看几个文件,判断命名 convention ===
# PPR10K 源文件常见命名: 0_0.tif, 0_1.tif ... 0_4.tif (5 个 augmentation)
# 无增强的就是 _0 后缀

import glob

for sub in sorted(os.listdir(PPR_ROOT)):
    full = f'{PPR_ROOT}/{sub}'
    if os.path.isdir(full):
        sample = sorted(os.listdir(full))[:5]
        last = sorted(os.listdir(full))[-3:]
        print(f"📂 {sub}/")
        print(f"   头 5: {sample}")
        print(f"   尾 3: {last}\n")

In [ ]:
# === Cell 3: 整理 paired 结构 (用 symlink,零空间) ===
# 标准做法: train = source + target_<expert>, 前 8875 张训练 / 后 2286 张验证
import os, glob, re

EXPERT = 'a'  # 选 a / b / c — 大多数论文用 a

# Cell 1 输出会告诉你 source 和 target 实际叫啥名,在这里调整:
# 常见命名 (以 PPR10K 标准 release 为准):
SOURCE_DIR = f'{PPR_ROOT}/source'         # ← 看 Cell 1 输出后改
TARGET_DIR = f'{PPR_ROOT}/target_{EXPERT}'  # ← 看 Cell 1 输出后改

if not (os.path.exists(SOURCE_DIR) and os.path.exists(TARGET_DIR)):
    print(f"❌ source 或 target_{EXPERT} 路径不对,你看 Cell 1 输出后改这个 cell 里的 SOURCE_DIR / TARGET_DIR")
    print(f"   SOURCE_DIR = {SOURCE_DIR} 存在? {os.path.exists(SOURCE_DIR)}")
    print(f"   TARGET_DIR = {TARGET_DIR} 存在? {os.path.exists(TARGET_DIR)}")
else:
    DEST = f'/content/drive/MyDrive/datasets/PPR10K_{EXPERT}'
    for sub in ['train/input', 'train/gt', 'val/input', 'val/gt']:
        os.makedirs(f'{DEST}/{sub}', exist_ok=True)

    # 拿出所有 source 里 "无增强" 的文件 (一般 _0 后缀,或者直接 N.tif)
    # 先看一眼实际命名 — Cell 2 输出有提示。下面假设格式 "<id>_0.tif"
    # 如果你的实际命名是 <id>.tif (无增强后缀),改 pattern 即可
    source_files = sorted([f for f in os.listdir(SOURCE_DIR) if f.endswith('_0.tif') or f.endswith('_0.tiff')])
    if not source_files:
        # fallback: 没有 _0 后缀就当所有 .tif 都是 source
        source_files = sorted([f for f in os.listdir(SOURCE_DIR) if f.lower().endswith(('.tif','.tiff'))])
    print(f"找到 {len(source_files)} 个 source 文件")

    target_files_set = set(os.listdir(TARGET_DIR))

    paired = []
    for sf in source_files:
        # PPR10K 配对规则: target 文件名通常去掉 source 的 _0 后缀,或者就是 source 的 base id
        candidate1 = sf  # 完全同名
        candidate2 = sf.replace('_0.tif', '.tif').replace('_0.tiff', '.tiff')  # 去掉 _0
        if candidate1 in target_files_set:
            paired.append((sf, candidate1))
        elif candidate2 in target_files_set:
            paired.append((sf, candidate2))
    print(f"成功配对 {len(paired)} 对")

    # 标准 PPR10K split: 前 8875 train / 后 2286 val
    train_pairs = paired[:8875]
    val_pairs = paired[8875:]
    print(f"Split: train={len(train_pairs)}, val={len(val_pairs)}")

    def link(src, dst):
        if not os.path.exists(dst):
            try: os.symlink(src, dst)
            except FileExistsError: pass

    for sf, tf in train_pairs:
        link(f'{SOURCE_DIR}/{sf}', f'{DEST}/train/input/{sf}')
        link(f'{TARGET_DIR}/{tf}', f'{DEST}/train/gt/{sf}')  # gt 用 source 的文件名,保证 paired_folder 能匹配
    for sf, tf in val_pairs:
        link(f'{SOURCE_DIR}/{sf}', f'{DEST}/val/input/{sf}')
        link(f'{TARGET_DIR}/{tf}', f'{DEST}/val/gt/{sf}')

    print(f"\n✅ {DEST} 准备好了")
    for sub in ['train/input', 'train/gt', 'val/input', 'val/gt']:
        n = len(os.listdir(f'{DEST}/{sub}'))
        print(f"   {sub}: {n} files")

In [ ]:
# === Cell 4: 验证 dataset 能加载,且 input != gt ===
import sys
sys.path.insert(0, '/content/drive/MyDrive/LoR-LUT')
from data.paired_folder import PairedFolderDataset

DEST = f'/content/drive/MyDrive/datasets/PPR10K_{EXPERT}'
ds = PairedFolderDataset(
    root=DEST,
    split='train',
    in_dir='input',
    gt_dir='gt',
    exts=('.tif', '.tiff'),
    patch=0,
    augment=False
)
print(f"✅ {len(ds)} train pairs")
for i in [0, 1, 2, len(ds)//2, len(ds)-1]:
    s = ds[i]
    diff = (s['img_in'] - s['img_gt']).abs().mean().item()
    status = '✅' if diff > 0.01 else '⚠️ input==gt?'
    print(f"  [{i:5d}] {s['name']:30s} input-gt MAE={diff:.4f} {status}")

## 完成后训练命令

```python
# 在主 quickstart notebook 里训练 PPR10K:
!python train.py \
    --cfg config/default.yaml \
    --data.root /content/drive/MyDrive/datasets/PPR10K_a \
    --work_dir /content/drive/MyDrive/LoR-LUT/runs/ppr10k_a_K0_R8
```

## 故障排除

**Q: Cell 1 报 PPR_ROOT 不存在?**  
A: shortcut 没加,或加到错地方了。再去 Drive 网页确认 `MyDrive/datasets/train_val_images_tif_360p` 在那。

**Q: Cell 3 显示 source/target 路径不对?**  
A: 看 Cell 1 / Cell 2 输出,把 `SOURCE_DIR` / `TARGET_DIR` 改成实际的子目录名。PPR10K 实际可能用 `train_input` / `train_target_a` 之类的命名。

**Q: Cell 3 只配对了 0 对?**  
A: source 和 target 的命名不匹配。看 Cell 2 输出的样本文件名,改 Cell 3 里的配对规则。

**Q: shortcut 加了但 Colab 看不到?**  
A: shortcut 同步到 Colab 可能有几分钟延迟。重新挂载 Drive (`drive.mount('/content/drive', force_remount=True)`)。